In [ ]:
import nbtlib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import json
from nbtlib.tag import Compound, List, Byte, Int, Short

def load_nbt(file_path):
    """Загружает .nbt файл и возвращает его содержимое."""
    try:
        nbt = nbtlib.load(file_path)
        return nbt
    except Exception as e:
        raise Exception(f"Ошибка при загрузке файла {file_path}: {e}")

def get_tag_value(tag):
    """Извлекает значение из NBT-тега, обрабатывая разные типы."""
    if isinstance(tag, (Int, Short, Byte)):
        value = int(tag)
        if isinstance(tag, Byte):
            value = value & 0xFF  
        return value
    elif isinstance(tag, (int, np.integer)):
        return tag
    else:
        raise ValueError(f"Неподдерживаемый тип тега: {type(tag)}")

def extract_block_name(compound):
    """Извлекает имя блока из Compound, включая свойства."""
    name = str(compound.get('Name', 'unknown'))
    if 'Properties' in compound:
        props = compound['Properties']
        props_str = ','.join([f"{k}={v}" for k, v in sorted(props.items())])
        return f"{name}[{props_str}]"
    return name

def create_palette_map(nbt):
    """Создает палитру из .nbt файла, сохраняя свойства блоков и резервируя 0 для воздуха."""
    if 'palette' not in nbt:
        raise KeyError("Файл не содержит тег 'palette'")

    palette = nbt['palette']
    palette_items = {}  
    reverse_palette = {}  
    
       air_found = False
    for idx, compound in enumerate(palette):
        block_name = extract_block_name(compound)
        if block_name == "minecraft:air":
            palette_items[block_name] = 0
            reverse_palette[0] = block_name
            air_found = True
            print(f"Резервирован блок для воздуха: {block_name} -> индекс 0")
            break
    if not air_found:
        palette_items["minecraft:air"] = 0
        reverse_palette[0] = "minecraft:air"
        print(f"Добавлен отсутствующий воздух: minecraft:air -> индекс 0")

  
    current_idx = 1
    for idx, compound in enumerate(palette):
        block_name = extract_block_name(compound)
        if block_name != "minecraft:air":  
            if block_name not in palette_items:
                palette_items[block_name] = current_idx
                reverse_palette[current_idx] = block_name
                print(f"Добавлен блок в палитру: {block_name} -> индекс {current_idx}")
                if block_name.startswith("minecraft:stone_bricks"):
                    stone_bricks_variants.append((block_name, current_idx))
                current_idx += 1


    block_entities = nbt.get('entities', [])
    print(f"BlockEntities (количество: {len(block_entities)}): {block_entities}")
    for entity in block_entities:
        entity_id = entity['id']
        if entity_id not in palette_items:
            palette_items[entity_id] = current_idx
            reverse_palette[current_idx] = entity_id
            print(f"Добавлен блок из BlockEntities: {entity_id} -> индекс {current_idx}")
            current_idx += 1

    return palette_items, reverse_palette

def extract_3d_arrays(nbt):
    """Извлекает данные блоков из .nbt файла структуры."""
    required_tags = ['size', 'blocks', 'palette']
    for tag in required_tags:
        if tag not in nbt:
            raise KeyError(f"Файл не содержит тег '{tag}'")

     width, height, length = [get_tag_value(x) for x in nbt['size']]

    offset = [0, 0, 0]
    print("Смещение структуры (Offset): X=0, Y=0, Z=0")

    blocks = nbt['blocks']
    data_version = get_tag_value(nbt.get('DataVersion', 0))
    print(f"DataVersion: {data_version}")

    palette_items, reverse_palette = create_palette_map(nbt)
    print(f"Итоговая палитра (первые 10): {dict(list(palette_items.items())[:10])}")

    block_data = [0] * (width * height * length)
    for block in blocks:
        pos = block['pos']
        x, y, z = [get_tag_value(coord) for coord in pos]
        state = get_tag_value(block['state'])
        index = x + (z + y * length) * width
        if index < len(block_data):
            block_data[index] = state + 1  

    expected_length = width * height * length
    print(f"Длина BlockData: {len(block_data)}, Ожидаемая длина: {expected_length}")

    valid_indices = set(palette_items.values())
    block_data_unique = set(block_data)
    unknown_indices = block_data_unique - valid_indices
    if unknown_indices:
        print(f"Обнаружены неизвестные индексы в BlockData: {unknown_indices}")
        for idx in unknown_indices:
            block_data = [0 if x == idx else x for x in block_data]  

    main_array = np.zeros((height, length, width), dtype=np.int32)
    for y in range(height):
        for z in range(length):
            for x in range(width):
                index = x + (z + y * length) * width
                if index < len(block_data):
                    block_id = block_data[index]
                    main_array[y, z, x] = block_id if block_id in valid_indices else 0

    shifted_main_array = np.zeros((height, length, width), dtype=np.int32)
    for y in range(height):
        for z in range(length):
            for x in range(width):
                new_x = x - offset[0]
                new_y = y - offset[1]
                new_z = z - offset[2]
                if (0 <= new_x < width) and (0 <= new_y < height) and (0 <= new_z < length):
                    shifted_main_array[y, z, x] = main_array[new_y, new_z, new_x]

    for y in range(height):
        non_zero_count = np.sum(shifted_main_array[y] != 0)
        print(f"Высота Y={y} (основной массив): {non_zero_count} ненулевых блоков")
        non_zero_coords = np.where(shifted_main_array[y] != 0)
        if len(non_zero_coords[0]) > 0:
            first_non_zero = (non_zero_coords[0][0], non_zero_coords[1][0])
            print(f"Первые ненулевые координаты на Y={y}: (Z={first_non_zero[0]}, X={first_non_zero[1]})")

    max_block_id = max(block_data) if block_data else 0
    print(f"Максимальный индекс в BlockData: {max_block_id}, Размер палитры: {len(palette_items)}")

    return shifted_main_array, palette_items, unknown_indices

def save_to_numpy(array, output_path):
    """Сохраняет 3D-массив в .npy файл."""
    np.save(output_path, array)
    print(f"3D-массив сохранен в {output_path}")

def visualize_3d_array(array, title, palette_map=None, unknown_indices=None, output_path=None):
    """Визуализирует 3D-массив как воксельный график."""
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    array = np.transpose(array, (2, 1, 0))
    voxel_array = array != 0

    unique_ids = np.unique(array[voxel_array])
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_ids)))
    color_map = {id: colors[i] for i, id in enumerate(unique_ids)}

    unknown_color = [0.5, 0.5, 0.5, 1]
    unknown_indices = unknown_indices or set()

    voxel_colors = np.zeros(array.shape + (4,), dtype=float)
    for x in range(array.shape[0]):
        for z in range(array.shape[1]):
            for y in range(array.shape[2]):
                if voxel_array[x, z, y]:
                    block_id = array[x, z, y]
                    if block_id in unknown_indices:
                        voxel_colors[x, z, y] = unknown_color
                    else:
                        voxel_colors[x, z, y] = color_map[block_id]

    ax.voxels(voxel_array, facecolors=voxel_colors, edgecolor='k')
    ax.set_xlabel('X (Width)')
    ax.set_ylabel('Z (Length)')
    ax.set_zlabel('Y (Height)')
    ax.set_title(title)

    max_range = max(array.shape)
    ax.set_box_aspect((array.shape[0] / max_range, array.shape[1] / max_range, array.shape[2] / max_range))
    ax.set_xlim(0, array.shape[0])
    ax.set_ylim(0, array.shape[1])
    ax.set_zlim(0, array.shape[2])

    if palette_map:
        reverse_palette = {v: k for k, v in palette_map.items()}
        legend_elements = [
            plt.Line2D([0], [0], marker='s', color='w', label=reverse_palette.get(id, f'ID {id}'),
                       markerfacecolor=color_map[id], markersize=10)
            for id in unique_ids if id in color_map and id not in unknown_indices
        ]
        if unknown_indices:
            legend_elements.append(
                plt.Line2D([0], [0], marker='s', color='w', label='Unknown Blocks',
                           markerfacecolor=unknown_color, markersize=10)
            )
        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.1, 1), prop={'size': 8})

    plt.subplots_adjust(right=0.85, top=0.95)
    if output_path:
        plt.savefig(output_path)
        print(f"Визуализация сохранена как {output_path}")
    plt.show()

def main():
    output_base_dir = "output"
    os.makedirs(output_base_dir, exist_ok=True)

    nbt_files = [f for f in os.listdir('.') if f.endswith('.nbt')]

    if not nbt_files:
        print("В текущей директории не найдено файлов с расширением .nbt")
        return

    for nbt_file in nbt_files:
        try:
            base_name = os.path.splitext(nbt_file)[0]
            output_dir = os.path.join(output_base_dir, base_name)
            os.makedirs(output_dir, exist_ok=True)
            npy_main_output = os.path.join(output_dir, f"{base_name}_main.npy")
            palette_output = os.path.join(output_dir, f"{base_name}_palette.json")
            png_main_output = os.path.join(output_dir, f"{base_name}_main.png")

            nbt = load_nbt(nbt_file)
            print(f"Обработка файла {nbt_file}. Доступные теги: {list(nbt.keys())}")
            main_array, palette_map, unknown_indices = extract_3d_arrays(nbt)
            save_to_numpy(main_array, npy_main_output)

            # Сохранение палитры в JSON (index -> block_name), начиная с 0 для воздуха
            reverse_palette = {str(idx): block_name for block_name, idx in palette_map.items()}
            with open(palette_output, 'w') as f:
                json.dump(reverse_palette, f, indent=4)
            print(f"Палитра сохранена как {palette_output}")

            print(f"Размеры основного массива для {nbt_file}: {main_array.shape}")
            print(f"Пример данных (первые 2 слоя): {main_array[:2, :2, :2]}")

            visualize_3d_array(main_array, '3D Визуализация основного .nbt файла', palette_map, unknown_indices, png_main_output)

        except Exception as e:
            print(f"Произошла ошибка при обработке {nbt_file}: {e}")
            if isinstance(e, KeyError):
                print("Проверьте структуру .nbt файла. Требуемые теги: size, blocks, palette")
                print(f"Доступные теги в файле: {list(nbt.keys())}")
            elif isinstance(e, ValueError):
                print("Проверьте типы данных тегов")

if __name__ == "__main__":
    main()